# Milestone 1


---

---

In [14]:
import re
import itertools
from collections import Counter

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [18]:
# ── Question 1 ────────────────────────────────────────────────────────────
# Frequency distribution of the correct answer, using Counter instead of
# value_counts().
answer_counts = dict(sorted(Counter(train['answer']).items()))
print(answer_counts)

sum_result = max(answer_counts.values()) + min(answer_counts.values())
print(sum_result)

{'A': 369, 'B': 490, 'C': 459, 'D': 358, 'E': 324}
814


In [19]:
# ── Question 2 ────────────────────────────────────────────────────────────
# Clean text with a regex substitution instead of str.translate, and flatten
# words with itertools.chain instead of a manual set-update loop.
def clean_text(text: str) -> str:
    return re.sub(r'[^\w\s]', '', text.lower())

cleaned_prompts = train['prompt'].apply(clean_text)
all_words = itertools.chain.from_iterable(p.split() for p in cleaned_prompts)
vocabulary_size = len(set(all_words))
print(vocabulary_size)

858


In [20]:
# ── Question 3 ────────────────────────────────────────────────────────────
# Filter stop words with filter()/lambda instead of a list comprehension.
row1_words = cleaned_prompts.iloc[0].split()
filtered_words = list(filter(lambda w: w not in ENGLISH_STOP_WORDS, row1_words))
remaining_words_count = len(filtered_words)
print(remaining_words_count)
print(filtered_words)

13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [21]:
# ── Question 4 ────────────────────────────────────────────────────────────
# Combine prompt + options row-wise with DataFrame.apply instead of Series
# concatenation with '+'.
def combine_row(row) -> str:
    return ' '.join([row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']])

combined_texts = train.apply(combine_row, axis=1).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_texts)
tfidf_vocab_size = X.shape[1]
print(tfidf_vocab_size)

2762


In [22]:
# ── Question 5 ────────────────────────────────────────────────────────────
# Transform prompt and option A together in a single call, then slice.
row1 = train.iloc[0]
vecs = vectorizer.transform([row1['prompt'], row1['A']])
sim_score = cosine_similarity(vecs[0], vecs[1])[0, 0]
print(f"{sim_score:.4f}")

0.2720


In [23]:
# ── Question 6 ────────────────────────────────────────────────────────────
# Transform prompt + all 5 options at once per row, use np.argmax instead of
# building a dict and calling max(..., key=...).
OPTIONS = ['A', 'B', 'C', 'D', 'E']
correct_flags = []

for _, row in train.iterrows():
    texts = [row['prompt']] + [row[opt] for opt in OPTIONS]
    vecs = vectorizer.transform(texts)
    sims = cosine_similarity(vecs[0], vecs[1:]).flatten()
    predicted_answer = OPTIONS[np.argmax(sims)]
    correct_flags.append(predicted_answer == row['answer'])

correct_count = sum(correct_flags)
total_count = len(train)
percentage = (correct_count / total_count) * 100
print(f"Correct predictions: {correct_count}/{total_count}")
print(f"Percentage: {percentage:.2f}%")

Correct predictions: 271/2000
Percentage: 13.55%


In [24]:
# ── Questions 7 & 8 ───────────────────────────────────────────────────────
# Single reusable MAP@3 function instead of duplicated inline logic.
def map_at_3(ground_truth: str, prediction: list) -> float:
    top3 = prediction[:3]
    return 1.0 / (top3.index(ground_truth) + 1) if ground_truth in top3 else 0.0

print(map_at_3('C', ['C', 'A', 'B']))   
print(map_at_3('B', ['D', 'B', 'E']))   

1.0
0.5


In [25]:
# ── Question 9 ────────────────────────────────────────────────────────────
# Majority Class Baseline, using Counter.most_common instead of sort_values.
freq = Counter(train['answer'])
top_answers = [ans for ans, _ in freq.most_common(3)]

majority_baseline_map3 = train['answer'].apply(
    lambda ans: map_at_3(ans, top_answers)
).mean()

print("Top 3 static prediction order:", top_answers)
print("Majority class baseline MAP@3:", majority_baseline_map3)

Top 3 static prediction order: ['B', 'C', 'A']
Majority class baseline MAP@3: 0.42125


In [26]:
# ── Question 10 ───────────────────────────────────────────────────────────
# TF-IDF pipeline: batch-transform prompt + options per row, rank with
# np.argsort instead of sorting (option, score) tuples, and reuse map_at_3.
map3_scores = []

for _, row in train.iterrows():
    texts = [row['prompt']] + [row[opt] for opt in OPTIONS]
    vecs = vectorizer.transform(texts)
    sims = cosine_similarity(vecs[0], vecs[1:]).flatten()
    ranked_answers = [OPTIONS[i] for i in np.argsort(-sims)]
    map3_scores.append(map_at_3(row['answer'], ranked_answers))

tfidf_pipeline_map3 = np.mean(map3_scores)
print(tfidf_pipeline_map3)

0.2915
